In [69]:
%run functions_dbs.py
import re
fs = 12
%matplotlib

Using matplotlib backend: MacOSX


In [50]:
# load measurement file with all measurements included
file = 'dataPrep/2021-10_Burggraben_EP_H2S.xls'
SoftwareFile = True

# get calibration from measurement file
dsheets = pd.read_excel(file, sheet_name=None)

In [121]:
h2s_para = dsheets['Calibration'].T.dropna()

In [117]:
sensors = dsheets['Sensors'][['Sensor', 'Type']]
ls_sens = list()
[ls_sens.append(i) for i in sensors['Sensor'].values if isinstance(i, int)]

ls_sensT = [sensors[sensors['Sensor'] == s]['Type'].values[0] for s in ls_sens]

In [161]:
n, ls_col = 0, list()
for en, i in enumerate(dsheets['Profiles'].columns):
    if 'Sensor' in i:
        ls_col.append(en)
        n+=1

# get the individual sensor profiles and link them to the sensor tyoe
dic_sensor = dict()
for en, s in enumerate(ls_sensT):
    if en+1 >= len(ls_sensT):
        l = dsheets['Profiles'].columns
        df = dsheets['Profiles'][dsheets['Profiles'].columns[ls_col[en]:len(l)]]
    else:
        df = dsheets['Profiles'][dsheets['Profiles'].columns[ls_col[en]:ls_col[en+1]]]
    df = df.T.set_index(0).T
    df.index = np.arange(0, len(df))
    dic_sensor[s] = df.dropna()

# split into cores and label with different sample-ID

In [163]:
for t in h2s_para.index:
    if 'Time' in t:
        ind_name = t

profiles_calib0 = dic_sensor['H2S'][dic_sensor['H2S']['Time'] <= h2s_para.loc[ind_name][1]]

In [188]:
# linear regression: m*S[µM] + t = H2S[mV] --> H2S[µM] = (H2S[mV] - t) / m
para0 = (h2s_para.T.loc[0].T.filter(like='Slope').values[0],
         h2s_para.T.loc[0].T.filter(like='Intercept').values[0])

In [190]:
(profiles_calib0['Signal (mV)'] + para0[1]) / para0[0]

0       2.517673
1       2.549987
2       2.570293
3       2.555209
4       2.573447
          ...   
1243    2.601838
1244    2.611063
1245    2.633542
1246    2.620818
1247    2.635928
Name: Signal (mV), Length: 1234, dtype: object